In [1]:
!pip install -q groq wikipedia-api sentence-transformers transformers

import os
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

import torch
print("GPU available:", torch.cuda.is_available())
print("Key loaded:", userdata.get("GROQ_API_KEY") is not None)

GPU available: True
Key loaded: True


In [11]:
import json, re

def get_llm_answer(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer the question in 4-6 factual sentences."},
            {"role": "user", "content": question}
        ],
        temperature=0.7,
        max_tokens=1000
    )
    return response.choices[0].message.content

def extract_claims(text):
    prompt = f"""Break the text below into independent, atomic factual claims.

Rules:
- One fact per claim.
- Replace pronouns (he, she, it, they) with the full name.
- Each claim must make sense on its own.
- Return ONLY a JSON list of strings, nothing else.

Text:
{text}"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1500
    )
    output = response.choices[0].message.content
    match = re.search(r"\[.*\]", output, re.DOTALL)
    return json.loads(match.group(0)) if match else []

question = "Tell me about the life and career of Rosalind Franklin."
answer = get_llm_answer(question)
claims = extract_claims(answer)

print("Number of claims:", len(claims))
for i, c in enumerate(claims):
    print(i, c)

Number of claims: 19
0 Rosalind Elsie Franklin was born in 1920.
1 Rosalind Elsie Franklin died in 1958.
2 Rosalind Elsie Franklin was a British chemist.
3 Rosalind Elsie Franklin was a British X‑ray crystallographer.
4 Rosalind Elsie Franklin's work was crucial to understanding the structure of DNA.
5 Rosalind Elsie Franklin earned a Ph.D. in physical chemistry from Cambridge.
6 Rosalind Elsie Franklin joined King's College London in 1951.
7 Rosalind Elsie Franklin produced the famous “Photograph 51,” an X‑ray diffraction image that revealed DNA’s helical form.
8 Rosalind Elsie Franklin moved to Birkbeck College in 1953.
9 Rosalind Elsie Franklin focused on the structure of viruses such as the tobacco mosaic virus at Birkbeck College.
10 Rosalind Elsie Franklin published pioneering papers on viral morphology.
11 Rosalind Elsie Franklin's contributions were not fully recognized during her lifetime.
12 James Watson cited Rosalind Elsie Franklin's data when he proposed the double‑helix m

In [12]:
import wikipediaapi
from sentence_transformers import SentenceTransformer, util

wiki = wikipediaapi.Wikipedia(user_agent="hallucination-detector/1.0 (student project)", language="en")
page = wiki.page("Rosalind Franklin")
print("Page exists:", page.exists())

embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", page.text)
             if len(s.strip()) > 40 and not s.strip().startswith('"')]
sentence_embeddings = embedder.encode(sentences, convert_to_tensor=True)
print("Sentences in page:", len(sentences))

def get_evidence(claim, top_k=3):
    claim_embedding = embedder.encode(claim, convert_to_tensor=True)
    scores = util.cos_sim(claim_embedding, sentence_embeddings)[0]
    top = scores.topk(top_k)
    return [(sentences[i], round(float(s), 3)) for s, i in zip(top.values, top.indices)]

for idx in [0, 1, 2]:
    print("\nCLAIM:", claims[idx])
    for sent, score in get_evidence(claims[idx]):
        print(f"  [{score}] {sent[:200]}")

Page exists: True


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentences in page: 464

CLAIM: Rosalind Elsie Franklin was born in 1920.
  [0.798] Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer.
  [0.667] Rosalind's middle name, "Elsie", was in memory of Hugh's first wife, who died in the 1918 flu pandemic.
  [0.63] The inscription on her tombstone reads:

IN MEMORY OF  ROSALIND ELSIE FRANKLIN  מ רחל בת ר׳ יהודה [Hebrew: m[arat] Rahel bat r' Yehuda, "M[s] Rachel daughter of R[eb] Judah"] DEARLY LOVED ELDER DAUGHT

CLAIM: Rosalind Elsie Franklin died in 1958.
  [0.704] Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer.
  [0.688] Rosalind's middle name, "Elsie", was in memory of Hugh's first wife, who died in the 1918 flu pandemic.
  [0.635] The inscription on her tombstone reads:

IN MEMORY OF  ROSALIND ELSIE FRANKLIN  מ רחל בת ר׳ יהודה [Hebrew: m[arat] Rahel bat r' Yehuda, "M[s] Rachel daughter of R[eb] Judah"] DEARLY LOVED ELDER DAUGHT


In [13]:
from transformers import pipeline

nli = pipeline("text-classification",
               model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
               top_k=None,
               device=0)

def verify_claim(claim, evidence_list):
    best = {"label": "neutral", "score": 0.0, "evidence": ""}
    for sent, _ in evidence_list:
        out = nli({"text": sent, "text_pair": claim})
        out = out[0] if isinstance(out[0], list) else out
        scores = {o["label"].lower(): o["score"] for o in out}
        ent, con = scores.get("entailment", 0), scores.get("contradiction", 0)
        if ent >= con and ent > best["score"]:
            best = {"label": "entailment", "score": ent, "evidence": sent}
        elif con > ent and con > best["score"]:
            best = {"label": "contradiction", "score": con, "evidence": sent}
    return best

for idx in [0, 1, 2]:
    result = verify_claim(claims[idx], get_evidence(claims[idx]))
    print(claims[idx])
    print("  ->", result["label"], round(result["score"], 2), "\n")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Rosalind Elsie Franklin was born in 1920.
  -> entailment 1.0 

Rosalind Elsie Franklin died in 1958.
  -> entailment 1.0 

Rosalind Elsie Franklin was a British chemist.
  -> entailment 0.96 



In [14]:
fake_claims = [
    "Rosalind Franklin was born in 1925.",
    "Rosalind Franklin won the Nobel Prize in Chemistry in 1962.",
    "Rosalind Franklin was an American physicist.",
    "Rosalind Franklin discovered penicillin."
]

for c in fake_claims:
    result = verify_claim(c, get_evidence(c))
    print(c)
    print("  ->", result["label"], round(result["score"], 2))
    print("  evidence:", result["evidence"][:150], "\n")

Rosalind Franklin was born in 1925.
  -> contradiction 0.99
  evidence: Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer. 

Rosalind Franklin won the Nobel Prize in Chemistry in 1962.
  -> contradiction 1.0
  evidence: Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer. 

Rosalind Franklin was an American physicist.
  -> contradiction 0.99
  evidence: Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer. 

Rosalind Franklin discovered penicillin.
  -> contradiction 0.93
  evidence: Rosalind Franklin and the discovery of DNA". 



In [15]:
import pandas as pd

def analyze(claims_list):
    rows = []
    for c in claims_list:
        r = verify_claim(c, get_evidence(c))
        verdict = {"entailment": "Supported", "contradiction": "Contradicted", "neutral": "Not enough info"}[r["label"]]
        rows.append({"claim": c, "verdict": verdict, "confidence": round(r["score"], 2), "evidence": r["evidence"][:120]})
    return pd.DataFrame(rows)

df = analyze(claims)
print(df[["claim", "verdict", "confidence"]].to_string())

total = len(df)
contradicted = (df["verdict"] == "Contradicted").sum()
unverified = (df["verdict"] == "Not enough info").sum()
print("\nTotal claims:", total)
print("Contradiction rate:", round(contradicted / total, 2))
print("Hallucination rate (contradicted + unverified):", round((contradicted + unverified) / total, 2))

                                                                                                                        claim       verdict  confidence
0                                                                                   Rosalind Elsie Franklin was born in 1920.     Supported        1.00
1                                                                                       Rosalind Elsie Franklin died in 1958.     Supported        1.00
2                                                                              Rosalind Elsie Franklin was a British chemist.     Supported        0.96
3                                                               Rosalind Elsie Franklin was a British X‑ray crystallographer.     Supported        0.96
4                                           Rosalind Elsie Franklin's work was crucial to understanding the structure of DNA.     Supported        0.58
5                                                Rosalind Elsie Franklin earned a Ph.D. 

In [16]:
!pip install -q datasets
from datasets import load_dataset
ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
print(ds)
print(ds["evaluation"][0].keys())

DatasetDict({
    evaluation: Dataset({
        features: ['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'],
        num_rows: 238
    })
})
dict_keys(['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'])


In [17]:
def verify_claim(claim, evidence_list, thr=0.8):
    best_ent = {"score": 0.0, "evidence": ""}
    best_con = {"score": 0.0, "evidence": ""}
    for sent, _ in evidence_list:
        out = nli({"text": sent, "text_pair": claim})
        out = out[0] if isinstance(out[0], list) else out
        scores = {o["label"].lower(): o["score"] for o in out}
        if scores.get("entailment", 0) > best_ent["score"]:
            best_ent = {"score": scores["entailment"], "evidence": sent}
        if scores.get("contradiction", 0) > best_con["score"]:
            best_con = {"score": scores["contradiction"], "evidence": sent}

    if best_ent["score"] >= thr:
        return {"label": "entailment", "score": best_ent["score"], "evidence": best_ent["evidence"]}
    if best_con["score"] >= thr:
        return {"label": "contradiction", "score": best_con["score"], "evidence": best_con["evidence"]}
    return {"label": "neutral", "score": max(best_ent["score"], best_con["score"]), "evidence": best_ent["evidence"]}

def analyze(claims_list):
    rows = []
    for c in claims_list:
        r = verify_claim(c, get_evidence(c, top_k=5))
        verdict = {"entailment": "Supported", "contradiction": "Contradicted", "neutral": "Not enough info"}[r["label"]]
        rows.append({"claim": c, "verdict": verdict, "confidence": round(r["score"], 2), "evidence": r["evidence"][:150]})
    return pd.DataFrame(rows)

df = analyze(claims)
print(df[["claim", "verdict", "confidence"]].to_string())

total = len(df)
contradicted = (df["verdict"] == "Contradicted").sum()
unverified = (df["verdict"] == "Not enough info").sum()
print("\nTotal claims:", total)
print("Contradiction rate:", round(contradicted / total, 2))
print("Hallucination rate (contradicted + unverified):", round((contradicted + unverified) / total, 2))

                                                                                                                        claim          verdict  confidence
0                                                                                   Rosalind Elsie Franklin was born in 1920.        Supported        1.00
1                                                                                       Rosalind Elsie Franklin died in 1958.        Supported        1.00
2                                                                              Rosalind Elsie Franklin was a British chemist.        Supported        0.96
3                                                               Rosalind Elsie Franklin was a British X‑ray crystallographer.        Supported        0.96
4                                           Rosalind Elsie Franklin's work was crucial to understanding the structure of DNA.  Not enough info        0.58
5                                                Rosalind Elsie Frankl

In [18]:
for idx in [5, 6, 16]:
    claim = df.loc[idx, "claim"]
    print("CLAIM:", claim)
    for sent, sim in get_evidence(claim, top_k=5):
        out = nli({"text": sent, "text_pair": claim})
        out = out[0] if isinstance(out[0], list) else out
        s = {o["label"].lower(): round(o["score"], 2) for o in out}
        print(f"  ent={s.get('entailment')} con={s.get('contradiction')} | {sent[:160]}")
    print()

CLAIM: Rosalind Elsie Franklin earned a Ph.D. in physical chemistry from Cambridge.
  ent=0.0 con=0.0 | Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer.
  ent=0.46 con=0.01 | Franklin graduated in 1941 with an undergraduate degree in natural sciences from Newnham College, Cambridge, and then enrolled for a PhD degree in physical chem
  ent=0.0 con=0.0 | Franklin was awarded a research fellowship at Newnham College, with which she joined the physical chemistry laboratory of the University of Cambridge to work un
  ent=0.0 con=0.0 | Cambridge and World War II
Franklin went to Newnham College, Cambridge, in 1938 and studied chemistry within the Natural Sciences Tripos.
  ent=0.0 con=0.0 | Publications
Rosalind Franklin's most notable publications are listed below.

CLAIM: Rosalind Elsie Franklin joined King's College London in 1951.
  ent=0.0 con=0.99 | King's College London
In 1950, Franklin was granted a three-year Turner & Newall

In [19]:
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", page.text)
             if len(s.strip()) > 40 and not s.strip().startswith('"')]
sentence_embeddings = embedder.encode(sentences, convert_to_tensor=True)
print("Sentences in page:", len(sentences))

def verify_claim(claim, evidence_list, thr=0.8, con_top=2, min_sim=0.5):
    best_ent = {"score": 0.0, "evidence": ""}
    best_con = {"score": 0.0, "evidence": ""}
    for rank, (sent, sim) in enumerate(evidence_list):
        out = nli({"text": sent, "text_pair": claim})
        out = out[0] if isinstance(out[0], list) else out
        scores = {o["label"].lower(): o["score"] for o in out}
        if scores.get("entailment", 0) > best_ent["score"]:
            best_ent = {"score": scores["entailment"], "evidence": sent}
        if rank < con_top and sim >= min_sim and scores.get("contradiction", 0) > best_con["score"]:
            best_con = {"score": scores["contradiction"], "evidence": sent}

    if best_ent["score"] >= thr:
        return {"label": "entailment", "score": best_ent["score"], "evidence": best_ent["evidence"]}
    if best_con["score"] >= thr:
        return {"label": "contradiction", "score": best_con["score"], "evidence": best_con["evidence"]}
    return {"label": "neutral", "score": max(best_ent["score"], best_con["score"]), "evidence": best_ent["evidence"]}

df = analyze(claims)
print(df[["claim", "verdict", "confidence"]].to_string())

total = len(df)
contradicted = (df["verdict"] == "Contradicted").sum()
unverified = (df["verdict"] == "Not enough info").sum()
print("\nTotal claims:", total)
print("Contradiction rate:", round(contradicted / total, 2))
print("Hallucination rate (contradicted + unverified):", round((contradicted + unverified) / total, 2))

Sentences in page: 476
                                                                                                                        claim          verdict  confidence
0                                                                                   Rosalind Elsie Franklin was born in 1920.        Supported        1.00
1                                                                                       Rosalind Elsie Franklin died in 1958.        Supported        1.00
2                                                                              Rosalind Elsie Franklin was a British chemist.        Supported        0.96
3                                                               Rosalind Elsie Franklin was a British X‑ray crystallographer.        Supported        0.96
4                                           Rosalind Elsie Franklin's work was crucial to understanding the structure of DNA.  Not enough info        0.58
5                                              

In [20]:
def analyze(claims_list):
    rows = []
    for c in claims_list:
        r = verify_claim(c, get_evidence(c, top_k=5), con_top=1)
        verdict = {"entailment": "Supported", "contradiction": "Contradicted", "neutral": "Not enough info"}[r["label"]]
        rows.append({"claim": c, "verdict": verdict, "confidence": round(r["score"], 2), "evidence": r["evidence"][:150]})
    return pd.DataFrame(rows)

df = analyze(claims)
print(df["verdict"].value_counts())
print("\nContradicted claims:")
for _, row in df[df["verdict"] == "Contradicted"].iterrows():
    print(" -", row["claim"], "|", row["confidence"])

verdict
Not enough info    11
Supported           7
Contradicted        1
Name: count, dtype: int64

Contradicted claims:
 - Rosalind Elsie Franklin is a key figure in molecular biology. | 0.99


In [21]:
fake_claims = [
    "Rosalind Franklin was born in 1925.",
    "Rosalind Franklin won the Nobel Prize in Chemistry in 1962.",
    "Rosalind Franklin was an American physicist.",
    "Rosalind Franklin discovered penicillin."
]

for c in fake_claims:
    r = verify_claim(c, get_evidence(c, top_k=5), con_top=1)
    print(c)
    print("  ->", r["label"], round(r["score"], 2), "\n")

Rosalind Franklin was born in 1925.
  -> contradiction 0.99 

Rosalind Franklin won the Nobel Prize in Chemistry in 1962.
  -> contradiction 1.0 

Rosalind Franklin was an American physicist.
  -> contradiction 0.99 

Rosalind Franklin discovered penicillin.
  -> contradiction 0.93 



In [22]:
from collections import Counter

ex = ds["evaluation"][0]
print("GPT-3 TEXT:\n", ex["gpt3_text"][:400], "\n")
print("REAL WIKIPEDIA:\n", ex["wiki_bio_text"][:400], "\n")

print("SENTENCES AND LABELS:")
for s, label in zip(ex["gpt3_sentences"], ex["annotation"]):
    print(f"  [{label}] {s[:100]}")

all_labels = [l for row in ds["evaluation"] for l in row["annotation"]]
print("\nTotal sentences in dataset:", len(all_labels))
print(Counter(all_labels))

GPT-3 TEXT:
 John Russell Reynolds (1820–1876) was an English lawyer, judge, and author. He was born in London, the son of a barrister, and was educated at Eton College and Trinity College, Cambridge. He was called to the bar in 1845, and became a Queen's Counsel in 1859. He was appointed a judge of the Court of Common Pleas in 1867, and was knighted in 1871.

Reynolds was a prolific author, writing on a wide  

REAL WIKIPEDIA:
 Sir John Russell Reynolds, 1st Baronet (22 May 1828 – 29 May 1896) was a British neurologist and physician. Reynolds was born in Romsey, Hampshire, as the son of John Reynolds, an independent minister, and the grandson of Dr. Henry Revell Reynolds. He received general education from his father, and was educated in his profession at University College, London, where he obtained three gold medals in 

SENTENCES AND LABELS:
  [major_inaccurate] John Russell Reynolds (1820–1876) was an English lawyer, judge, and author.
  [major_inaccurate] He was born in London, t

In [23]:
def get_evidence_from(claim, ev_sentences, ev_emb, top_k=5):
    ce = embedder.encode(claim, convert_to_tensor=True)
    scores = util.cos_sim(ce, ev_emb)[0]
    top = scores.topk(min(top_k, len(ev_sentences)))
    return [(ev_sentences[i], float(s)) for s, i in zip(top.values, top.indices)]

rows = []
for p in range(10):
    ex = ds["evaluation"][p]
    ev_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", ex["wiki_bio_text"]) if len(s.strip()) > 20]
    ev_emb = embedder.encode(ev_sents, convert_to_tensor=True)
    for sent, label in zip(ex["gpt3_sentences"], ex["annotation"]):
        r = verify_claim(sent, get_evidence_from(sent, ev_sents, ev_emb), con_top=1, min_sim=0.3)
        rows.append({"passage": p, "label": label, "verdict": r["label"], "conf": round(r["score"], 2)})

res = pd.DataFrame(rows)
print("Sentences tested:", len(res))
print(pd.crosstab(res["label"], res["verdict"]))

Sentences tested: 89
verdict           contradiction  entailment  neutral
label                                               
accurate                      1           1       11
major_inaccurate             12           1       32
minor_inaccurate             15           0       16


In [24]:
shown = 0
for p in range(10):
    ex = ds["evaluation"][p]
    ev_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", ex["wiki_bio_text"]) if len(s.strip()) > 20]
    ev_emb = embedder.encode(ev_sents, convert_to_tensor=True)
    for sent, label in zip(ex["gpt3_sentences"], ex["annotation"]):
        if label != "accurate" or shown >= 8:
            continue
        top = get_evidence_from(sent, ev_sents, ev_emb, top_k=1)[0]
        out = nli({"text": top[0], "text_pair": sent})
        out = out[0] if isinstance(out[0], list) else out
        s = {o["label"].lower(): round(o["score"], 2) for o in out}
        print("SENTENCE:", sent[:120])
        print("  best evidence:", top[0][:160])
        print("  sim =", round(top[1], 2), "| ent =", s.get("entailment"), "| neutral =", s.get("neutral"), "| con =", s.get("contradiction"), "\n")
        shown += 1

SENTENCE: Mahler was drafted by the Braves in the first round of the 1975 amateur draft.
  best evidence: Mahler started on Opening Day for the Braves in 1982, when Atlanta won the National League West title.
  sim = 0.67 | ent = 0.0 | neutral = 0.99 | con = 0.01 

SENTENCE: He served as Commissioner from 1994 to 2016.
  best evidence: He took the position on June 1, 1994, succeeding Deane Beman, who retired after nearly 20 years in the post.
  sim = 0.49 | ent = 0.0 | neutral = 0.97 | con = 0.03 

SENTENCE: He was promoted to Deputy Commissioner in 1988 and Commissioner in 1994.
  best evidence: He took the position on June 1, 1994, succeeding Deane Beman, who retired after nearly 20 years in the post.
  sim = 0.44 | ent = 0.0 | neutral = 0.91 | con = 0.08 

SENTENCE: He also oversaw the introduction of the FedEx Cup, a season-long points competition that culminates in a four-tournament
  best evidence: As Commissioner, Finchem has greatly expanded the earnings opportunities for PGA T

In [25]:
import requests

def find_entity(question):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content":
            f"Return ONLY the name of the main person, place or thing this question is about, nothing else.\n\nQuestion: {question}"}],
        temperature=0,
        max_tokens=500
    )
    return r.choices[0].message.content.strip()

def find_wiki_page(name):
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "list": "search", "srsearch": name, "format": "json", "srlimit": 1},
        headers={"User-Agent": "hallucination-detector/1.0 (student project)"}
    )
    hits = resp.json()["query"]["search"]
    return hits[0]["title"] if hits else None

questions = [
    "Tell me about the life and career of Marie Curie.",
    "Tell me about Alan Turing.",
    "Tell me about the cricketer Sachin Tendulkar.",
    "Tell me about the Eiffel Tower.",
    "Tell me about the mathematician Ada Lovelace.",
]

for q in questions:
    name = find_entity(q)
    title = find_wiki_page(name)
    print(q, "->", name, "->", title)

Tell me about the life and career of Marie Curie. -> Marie Curie -> Marie Curie
Tell me about Alan Turing. -> Alan Turing -> Alan Turing
Tell me about the cricketer Sachin Tendulkar. -> Sachin Tendulkar -> Sachin Tendulkar
Tell me about the Eiffel Tower. -> Eiffel Tower -> Eiffel Tower
Tell me about the mathematician Ada Lovelace. -> Ada Lovelace -> Ada Lovelace


In [26]:
def run_pipeline(question):
    answer = get_llm_answer(question)
    claims_list = extract_claims(answer)
    title = find_wiki_page(find_entity(question))
    page_text = wiki.page(title).text

    ev_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", page_text)
                if len(s.strip()) > 40 and not s.strip().startswith('"')]
    ev_emb = embedder.encode(ev_sents, convert_to_tensor=True)

    rows = []
    for c in claims_list:
        r = verify_claim(c, get_evidence_from(c, ev_sents, ev_emb, top_k=5), con_top=1)
        verdict = {"entailment": "Supported", "contradiction": "Contradicted", "neutral": "Not enough info"}[r["label"]]
        rows.append({"claim": c, "verdict": verdict, "confidence": round(r["score"], 2), "evidence": r["evidence"][:150]})
    return title, pd.DataFrame(rows)

for q in ["Tell me about the life and career of Marie Curie.",
          "Tell me about Alan Turing.",
          "Tell me about the Eiffel Tower."]:
    title, out = run_pipeline(q)
    print("\n==", q, "| page:", title)
    print(out["verdict"].value_counts().to_string())
    for _, row in out[out["verdict"] == "Contradicted"].iterrows():
        print("  CONTRADICTED:", row["claim"])
        print("     evidence:", row["evidence"])


== Tell me about the life and career of Marie Curie. | page: Marie Curie
verdict
Not enough info    14
Supported          13

== Tell me about Alan Turing. | page: Alan Turing
verdict
Supported          10
Not enough info     7

== Tell me about the Eiffel Tower. | page: Eiffel Tower
verdict
Not enough info    12
Supported           6


In [27]:
def build_evidence(title):
    page_text = wiki.page(title).text
    ev_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", page_text)
                if len(s.strip()) > 40 and not s.strip().startswith('"')]
    ev_emb = embedder.encode(ev_sents, convert_to_tensor=True)
    return ev_sents, ev_emb

print("NOT ENOUGH INFO claims:")
for c in out[out["verdict"] == "Not enough info"]["claim"]:
    print(" -", c)

print("\nSUPPORTED claims (claim, then evidence):")
for _, r in out[out["verdict"] == "Supported"].head(4).iterrows():
    print(" -", r["claim"])
    print("     ", r["evidence"])

ev_sents, ev_emb = build_evidence("Eiffel Tower")
false_claims = [
    "The Eiffel Tower was completed in 1925.",
    "The Eiffel Tower is located in Berlin.",
    "The Eiffel Tower was designed by Leonardo da Vinci.",
    "The Eiffel Tower is 50 metres tall.",
]
print("\nFALSE CLAIMS TEST:")
for c in false_claims:
    r = verify_claim(c, get_evidence_from(c, ev_sents, ev_emb, top_k=5), con_top=1)
    print(" ", c, "->", r["label"], round(r["score"], 2))

NOT ENOUGH INFO claims:
 - The Eiffel Tower is a wrought‑iron lattice tower.
 - The Eiffel Tower stands 330 meters (1,083 feet) tall.
 - The Eiffel Tower is the tallest structure in Paris.
 - The Eiffel Tower has three levels that are accessible to the public.
 - Restaurants occupy the first level of the Eiffel Tower.
 - Restaurants occupy the second level of the Eiffel Tower.
 - An observation deck on the third level of the Eiffel Tower offers panoramic views of the city.
 - The Eiffel Tower was constructed from 18,038 iron pieces riveted together.
 - The Eiffel Tower requires regular repainting approximately every seven years to protect it from corrosion.
 - The Eiffel Tower was originally intended to be a temporary installation.
 - The Eiffel Tower became a permanent symbol of French engineering.
 - The Eiffel Tower is visited by more than 7 million tourists each year.

SUPPORTED claims (claim, then evidence):
 - The Eiffel Tower is located on the Champ de Mars in Paris, France.
   

In [28]:
tests = [
    "The Eiffel Tower is 50 metres tall.",
    "The Eiffel Tower was designed by Leonardo da Vinci.",
    "The Eiffel Tower stands 330 meters (1,083 feet) tall.",
    "The Eiffel Tower was constructed from 18,038 iron pieces riveted together.",
]
for c in tests:
    print("CLAIM:", c)
    for sent, sim in get_evidence_from(c, ev_sents, ev_emb, top_k=5):
        res_ = nli({"text": sent, "text_pair": c})
        res_ = res_[0] if isinstance(res_[0], list) else res_
        s = {o["label"].lower(): round(o["score"], 2) for o in res_}
        print(f"  sim={round(sim, 2)} ent={s.get('entailment')} con={s.get('contradiction')} | {sent[:140]}")
    print()

CLAIM: The Eiffel Tower is 50 metres tall.
  sim=0.76 ent=0.0 con=0.02 | Structures in France taller than the Eiffel Tower
  sim=0.74 ent=0.0 con=0.0 | The Tallest Tower: Eiffel and the Belle Epoque.
  sim=0.71 ent=0.0 con=0.0 | Excluding transmitters, the Eiffel Tower is the second tallest free-standing structure in France after the Millau Viaduct.
  sim=0.7 ent=0.0 con=0.0 | Lattice towers taller than the Eiffel Tower
  sim=0.7 ent=0.0 con=0.69 | The tower is 330 metres (1,083 ft) tall, about the same height as an 81-storey building, and the tallest structure in Paris.

CLAIM: The Eiffel Tower was designed by Leonardo da Vinci.
  sim=0.75 ent=0.0 con=0.03 | Eiffel's Tower: The Thrilling Story Behind Paris's Beloved Monument ….
  sim=0.73 ent=0.0 con=0.0 | As one of the most famous landmarks in the world, the Eiffel Tower has been the inspiration for the creation of many replicas and similar to
  sim=0.69 ent=0.0 con=1.0 | The design of the Eiffel Tower is attributed to Maurice Koechl

In [29]:
def build_evidence_clean(title):
    text = wiki.page(title).text
    sents = []
    for s in re.split(r"(?<=[.!?])\s+|\n+", text):
        s = s.strip()
        if len(s.split()) >= 8 and s.endswith(".") and ":" not in s:
            sents.append(s)
    return sents, embedder.encode(sents, convert_to_tensor=True)

ev_sents2, ev_emb2 = build_evidence_clean("Eiffel Tower")
print("Sentences kept:", len(ev_sents2), "(before:", len(ev_sents), ")\n")

tests = [
    "The Eiffel Tower is 50 metres tall.",
    "The Eiffel Tower was designed by Leonardo da Vinci.",
    "The Eiffel Tower stands 330 meters (1,083 feet) tall.",
    "The Eiffel Tower was constructed from 18,038 iron pieces riveted together.",
]
for c in tests:
    ev = get_evidence_from(c, ev_sents2, ev_emb2, top_k=5)
    r1 = verify_claim(c, ev, con_top=1)
    r3 = verify_claim(c, ev, con_top=3)
    print(c)
    print("  top-1 rule:", r1["label"], round(r1["score"], 2))
    print("  top-3 rule:", r3["label"], round(r3["score"], 2), "\n")

Sentences kept: 264 (before: 313 )

The Eiffel Tower is 50 metres tall.
  top-1 rule: neutral 0.0
  top-3 rule: neutral 0.69 

The Eiffel Tower was designed by Leonardo da Vinci.
  top-1 rule: neutral 0.0
  top-3 rule: contradiction 1.0 

The Eiffel Tower stands 330 meters (1,083 feet) tall.
  top-1 rule: neutral 0.1
  top-3 rule: neutral 0.1 

The Eiffel Tower was constructed from 18,038 iron pieces riveted together.
  top-1 rule: neutral 0.32
  top-3 rule: neutral 0.32 



In [30]:
height_sent = [s for s in ev_sents2 if "330 metres" in s][0]
print("EVIDENCE:", height_sent, "\n")

variants = [
    "The Eiffel Tower stands 330 meters (1,083 feet) tall.",
    "The Eiffel Tower is 330 metres (1,083 ft) tall.",
    "The Eiffel Tower is 330 metres tall.",
    "The Eiffel Tower is the tallest structure in Paris.",
]
for c in variants:
    out_ = nli({"text": height_sent, "text_pair": c})
    out_ = out_[0] if isinstance(out_[0], list) else out_
    s = {o["label"].lower(): round(o["score"], 2) for o in out_}
    print(f"ent={s.get('entailment')} neu={s.get('neutral')} con={s.get('contradiction')} | {c}")

EVIDENCE: The tower is 330 metres (1,083 ft) tall, about the same height as an 81-storey building, and the tallest structure in Paris. 

ent=0.06 neu=0.94 con=0.0 | The Eiffel Tower stands 330 meters (1,083 feet) tall.
ent=0.03 neu=0.97 con=0.0 | The Eiffel Tower is 330 metres (1,083 ft) tall.
ent=0.02 neu=0.98 con=0.0 | The Eiffel Tower is 330 metres tall.
ent=0.04 neu=0.96 con=0.0 | The Eiffel Tower is the tallest structure in Paris.


In [31]:
claim = "The Eiffel Tower is 330 metres (1,083 ft) tall."
long_tail = ", about the same height as an 81-storey building, and the tallest structure in Paris."

evidences = {
    "A. original (long, 'The tower')":        "The tower is 330 metres (1,083 ft) tall" + long_tail,
    "B. long, 'The Eiffel Tower'":            "The Eiffel Tower is 330 metres (1,083 ft) tall" + long_tail,
    "C. short, 'The tower'":                  "The tower is 330 metres (1,083 ft) tall.",
    "D. short, 'The Eiffel Tower' (sanity)":  "The Eiffel Tower is 330 metres (1,083 ft) tall.",
}
for name, ev in evidences.items():
    out_ = nli({"text": ev, "text_pair": claim})
    out_ = out_[0] if isinstance(out_[0], list) else out_
    s = {o["label"].lower(): round(o["score"], 2) for o in out_}
    print(f"ent={s.get('entailment')} neu={s.get('neutral')} con={s.get('contradiction')} | {name}")

ent=0.03 neu=0.97 con=0.0 | A. original (long, 'The tower')
ent=1.0 neu=0.0 con=0.0 | B. long, 'The Eiffel Tower'
ent=0.0 neu=1.0 con=0.0 | C. short, 'The tower'
ent=1.0 neu=0.0 con=0.0 | D. short, 'The Eiffel Tower' (sanity)


In [32]:
def resolve_subject(sent, title):
    last = title.split()[-1]
    s = re.sub(rf"^The {last}\b", f"The {title}", sent, flags=re.I)
    s = re.sub(r"^(He|She|It|They)\b", title, s)
    s = re.sub(r"^(His|Her|Its|Their)\b", f"{title}'s", s)
    return s

def build_evidence_resolved(title):
    text = wiki.page(title).text
    sents = []
    for s in re.split(r"(?<=[.!?])\s+|\n+", text):
        s = s.strip()
        if len(s.split()) >= 8 and s.endswith(".") and ":" not in s:
            sents.append(resolve_subject(s, title))
    return sents, embedder.encode(sents, convert_to_tensor=True)

ev_sents3, ev_emb3 = build_evidence_resolved("Eiffel Tower")

true_claims = [
    "The Eiffel Tower is a wrought-iron lattice tower.",
    "The Eiffel Tower stands 330 meters (1,083 feet) tall.",
    "The Eiffel Tower is the tallest structure in Paris.",
    "The Eiffel Tower has three levels that are accessible to the public.",
    "Restaurants occupy the first level of the Eiffel Tower.",
    "Restaurants occupy the second level of the Eiffel Tower.",
    "The Eiffel Tower was constructed from 18,038 iron pieces riveted together.",
    "The Eiffel Tower requires regular repainting approximately every seven years to protect it from corrosion.",
    "The Eiffel Tower was originally intended to be a temporary installation.",
    "The Eiffel Tower became a permanent symbol of French engineering.",
    "The Eiffel Tower is visited by more than 7 million tourists each year.",
]
false_claims = [
    "The Eiffel Tower was completed in 1925.",
    "The Eiffel Tower is located in Berlin.",
    "The Eiffel Tower was designed by Leonardo da Vinci.",
    "The Eiffel Tower is 50 metres tall.",
]

from collections import Counter
for rule in [1, 3]:
    print(f"\n=== contradiction rule: top-{rule} ===")
    t = [verify_claim(c, get_evidence_from(c, ev_sents3, ev_emb3, top_k=5), con_top=rule)["label"] for c in true_claims]
    print("True claims  ->", dict(Counter(t)))
    for c in false_claims:
        r = verify_claim(c, get_evidence_from(c, ev_sents3, ev_emb3, top_k=5), con_top=rule)
        print("  FALSE:", c, "->", r["label"], round(r["score"], 2))


=== contradiction rule: top-1 ===
True claims  -> {'neutral': 6, 'entailment': 5}
  FALSE: The Eiffel Tower was completed in 1925. -> contradiction 1.0
  FALSE: The Eiffel Tower is located in Berlin. -> contradiction 1.0
  FALSE: The Eiffel Tower was designed by Leonardo da Vinci. -> neutral 0.0
  FALSE: The Eiffel Tower is 50 metres tall. -> contradiction 1.0

=== contradiction rule: top-3 ===
True claims  -> {'neutral': 6, 'entailment': 5}
  FALSE: The Eiffel Tower was completed in 1925. -> contradiction 1.0
  FALSE: The Eiffel Tower is located in Berlin. -> contradiction 1.0
  FALSE: The Eiffel Tower was designed by Leonardo da Vinci. -> contradiction 1.0
  FALSE: The Eiffel Tower is 50 metres tall. -> contradiction 1.0
